# Vietnam Financial News–Market EDA
Notebook tổng hợp chạy tuần tự News Silver, Market Silver và Event–Market Gold. Mỗi cell chỉ hiển thị tối đa một bảng.

## News Silver EDA
EDA riêng cho chất lượng, nội dung, nguồn và ticker của Silver News.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
news = pd.read_parquet(ROOT / 'data_lake/silver/news')
news['published_timestamp'] = pd.to_datetime(news['published_timestamp'], utc=True)
news['published_month'] = news['published_timestamp'].dt.tz_convert('Asia/Ho_Chi_Minh').dt.to_period('M').astype(str)
print(f'News rows: {len(news):,}')

In [ ]:
quality = pd.Series({
    'rows': len(news),
    'distinct_event_id': news.event_id.nunique(),
    'duplicate_url': news.url.dropna().duplicated().sum(),
    'duplicate_content_hash': news.content_hash.dropna().duplicated().sum(),
    'invalid_rows': (~news.is_valid).sum(),
    'missing_title': news.title.fillna('').str.strip().eq('').sum(),
    'full_article': news.has_full_text.sum(),
    'metadata_only': (~news.has_full_text).sum(),
    'with_vn30_ticker': news.merged_tickers.map(len).gt(0).sum(),
}, name='value').to_frame()
display(quality)

In [ ]:
date_range = news[['published_timestamp']].agg(['min','max'])
display(date_range)

In [ ]:
content_status = news.has_full_text.map({True: 'Full article', False: 'Metadata only'}).value_counts()
full_lengths = news.loc[news.has_full_text, 'content_length']
display(pd.DataFrame({
    'articles': content_status,
    'rate_pct': (100 * content_status / len(news)).round(2),
}))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
content_status.plot.bar(ax=axes[0], title='Metadata-only và full article', rot=0)
full_lengths.plot.hist(bins=15, ax=axes[1], title='Độ dài chỉ các full article')
axes[1].set_xlabel('Số ký tự')
plt.tight_layout(); plt.show()

In [ ]:
full_length_summary = full_lengths.describe().to_frame('full_article_length')
display(full_length_summary)

In [ ]:
source_stats = news.groupby('source').agg(
    articles=('event_id','size'),
    full_articles=('has_full_text','sum'),
).sort_values('articles', ascending=False)
source_stats['full_text_rate_pct'] = (100 * source_stats.full_articles / source_stats.articles).round(2)
display(source_stats.head(20))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
source_stats.head(15).sort_values('articles').articles.plot.barh(ax=axes[0], title='Top 15 nguồn')
news.groupby('published_month').size().plot(ax=axes[1], title='Số bài theo tháng')
axes[1].tick_params(axis='x', rotation=75)
plt.tight_layout(); plt.show()

In [ ]:
ticker_counts = news.explode('merged_tickers').merged_tickers.value_counts()
ticker_pairs = [(list(r), list(n)) for r,n in zip(news.rulebased_tickers, news.ner_tickers)]
agreement = pd.Series({
    'rule_ner_agree': sum(set(r) == set(n) and len(r) > 0 for r,n in ticker_pairs),
    'rule_only': sum(len(r) > 0 and len(n) == 0 for r,n in ticker_pairs),
    'ner_only': sum(len(n) > 0 and len(r) == 0 for r,n in ticker_pairs),
    'partial_disagreement': sum(len(r) > 0 and len(n) > 0 and set(r) != set(n) for r,n in ticker_pairs),
})
display(agreement.to_frame('articles'))

In [ ]:
display(ticker_counts.to_frame('mentions'))
ticker_counts.sort_values().plot.barh(figsize=(9,8), title='Ticker VN30 được nhắc đến')
plt.tight_layout(); plt.show()

## Market Silver EDA
EDA riêng cho coverage, chất lượng OHLCV, return và volatility của VN30.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
market = pd.read_parquet(ROOT / 'data_lake/silver/market_daily')
market['session_date'] = pd.to_datetime(market.session_date.astype(str))
print(f'Market rows: {len(market):,}')

In [ ]:
quality = pd.Series({
    'rows': len(market),
    'tickers': market.ticker.nunique(),
    'sessions': market.session_date.nunique(),
    'first_session': market.session_date.min(),
    'last_session': market.session_date.max(),
    'duplicate_keys': market.duplicated(['ticker','session_date','interval','source']).sum(),
    'invalid_rows': (~market.is_valid).sum(),
    'weekend_rows': market.session_date.dt.dayofweek.ge(5).sum(),
    'missing_daily_return': market.daily_return.isna().sum(),
    'abs_return_gt_10pct': market.daily_return.abs().gt(.10).sum(),
}, name='value').to_frame()
display(quality)

In [ ]:
coverage = market.groupby('ticker').agg(
    sessions=('session_date','nunique'),
    first_session=('session_date','min'),
    last_session=('session_date','max'),
    missing_return=('daily_return', lambda s: s.isna().sum()),
).sort_values('sessions')
display(coverage)
coverage.sessions.plot.bar(figsize=(13,4), title='Số phiên theo ticker')
plt.ylabel('Phiên'); plt.tight_layout(); plt.show()

In [ ]:
returns = market.daily_return.dropna()
return_summary = returns.describe(percentiles=[.01,.05,.5,.95,.99]).to_frame('value')
return_summary['unit'] = '%'
return_summary.loc['count', 'unit'] = 'rows'
return_summary.loc[return_summary.index != 'count', 'value'] *= 100
display(return_summary.rename(columns={'value':'daily_return'}))

In [ ]:
ticker_risk = market.groupby('ticker').agg(
    mean_return=('daily_return','mean'),
    volatility=('daily_return','std'),
    median_volume=('volume','median'),
).sort_values('volatility', ascending=False)
display(ticker_risk.assign(mean_return=lambda x:100*x.mean_return, volatility=lambda x:100*x.volatility))
fig, axes = plt.subplots(1,2,figsize=(14,5))
(100*returns).clip(-15,15).plot.hist(bins=60, ax=axes[0], title='Daily return (%)')
(100*ticker_risk.volatility).sort_values().plot.barh(ax=axes[1], title='Volatility theo ticker (%)')
plt.tight_layout(); plt.show()

In [ ]:
return_matrix = market.pivot(index='session_date', columns='ticker', values='daily_return')
corr = return_matrix.corr(min_periods=100)
fig, ax = plt.subplots(figsize=(10,8))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.index)
ax.set_title('Tương quan daily return')
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()

## Event–Market Gold EDA
EDA riêng cho point-in-time join, event-window coverage và abnormal return.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
gold = pd.read_parquet(ROOT / 'data_lake/gold/news_market_impact')
for c in ['event_date_vn','t0_date','anchor_date_t_minus_1','session_date']:
    gold[c] = pd.to_datetime(gold[c], errors='coerce')
print(f'Gold rows: {len(gold):,}')

In [ ]:
pairs = gold.drop_duplicates(['event_id','ticker']).copy()
bad_t0 = np.where(pairs.published_after_close, pairs.t0_date <= pairs.event_date_vn, pairs.t0_date < pairs.event_date_vn)
quality = pd.Series({
    'rows': len(gold),
    'event_ticker_pairs': len(pairs),
    'events': gold.event_id.nunique(),
    'tickers': gold.ticker.nunique(),
    'duplicate_window_keys': gold.duplicated(['event_id','ticker','relative_session']).sum(),
    'published_after_close': pairs.published_after_close.sum(),
    'bad_t0': bad_t0.sum(),
    'bad_anchor': (pairs.anchor_date_t_minus_1 >= pairs.t0_date).sum(),
}, name='value').to_frame()
display(quality)

In [ ]:
coverage = gold.groupby('relative_session').agg(
    requested=('event_id','size'),
    available=('has_market_data','sum'),
)
coverage['availability_pct'] = (100*coverage.available/coverage.requested).round(2)
display(coverage)
coverage.availability_pct.plot.bar(figsize=(9,4), ylim=(0,105), title='Market coverage theo relative session (%)', rot=0)
plt.tight_layout(); plt.show()

In [ ]:
available = gold[gold.has_market_data & gold.abnormal_return.notna()].copy()
stats = available.groupby('relative_session').abnormal_return.agg(['count','mean','median','std'])
stats['se'] = stats['std']/np.sqrt(stats['count'])
stats['ci95_low'] = stats['mean'] - 1.96*stats['se']
stats['ci95_high'] = stats['mean'] + 1.96*stats['se']
display((100*stats).assign(count=stats['count']).round(3))
fig, ax = plt.subplots(figsize=(9,5))
x = stats.index.to_numpy()
mean = (100*stats['mean']).to_numpy()
error = (100*1.96*stats['se']).to_numpy()
ax.errorbar(x, mean, yerr=error, marker='o', capsize=4)
ax.axhline(0,color='black',linewidth=.8)
ax.set(title='Mean cumulative abnormal return', xlabel='Relative session', ylabel='%')
plt.tight_layout(); plt.show()

In [ ]:
focus = available[available.relative_session.isin([0,1,3,5])]
timing = focus.groupby(['published_after_close','relative_session']).agg(
    events=('event_id','size'), mean_abnormal=('abnormal_return','mean'),
).reset_index()
timing['mean_abnormal_pct'] = (100*timing.mean_abnormal).round(3)
display(timing)

In [ ]:
ticker_t5 = available[available.relative_session.eq(5)].groupby('ticker').agg(
    events=('event_id','size'), mean_abnormal=('abnormal_return','mean'),
).query('events >= 20').sort_values('mean_abnormal')
ticker_t5['mean_abnormal_pct'] = (100*ticker_t5.mean_abnormal).round(3)
display(ticker_t5[['events','mean_abnormal_pct']])
print('Cảnh báo: các thống kê này mô tả tương quan, không chứng minh tác động nhân quả.')